# 01 — Sales EDA

Quick exploratory pass over `analytics.fact_sales` joined to its dimensions.
Run this **after** `make full-refresh` (or `make bootstrap` on a fresh DB).

Connects via the `DB_*` variables in `.env`.

In [ ]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import create_engine, text

load_dotenv(Path('..') / '.env')

DB_URL = (
    f"postgresql+psycopg2://{os.environ['DB_USER']}:{os.environ['DB_PASSWORD']}"
    f"@{os.environ.get('DB_HOST', 'localhost')}:{os.environ.get('DB_PORT', 5432)}"
    f"/{os.environ['DB_NAME']}"
)
engine = create_engine(DB_URL)
engine

## Row counts

In [ ]:
tables = [
    'dim_date', 'dim_customer', 'dim_product', 'dim_store', 'dim_currency',
    'fact_sales', 'fact_weather_daily', 'fact_fx_rates',
]
with engine.connect() as conn:
    counts = {t: conn.execute(text(f'SELECT COUNT(*) FROM analytics.{t}')).scalar() for t in tables}
pd.Series(counts, name='rows').to_frame()

## Top-10 states by revenue (BRL)

In [ ]:
sql = '''
SELECT  dc.state                                AS customer_state,
        COUNT(DISTINCT fs.order_code)           AS orders,
        ROUND(SUM(fs.unit_price + fs.freight_value)::numeric, 2) AS revenue_brl
FROM    analytics.fact_sales fs
JOIN    analytics.dim_customer dc USING (customer_key)
GROUP BY dc.state
ORDER BY revenue_brl DESC
LIMIT   10;
'''
pd.read_sql(sql, engine)

## Monthly revenue trend

In [ ]:
sql = '''
SELECT  dd.year, dd.month,
        ROUND(SUM(fs.unit_price + fs.freight_value)::numeric, 2) AS revenue_brl,
        COUNT(*)                                AS line_items
FROM    analytics.fact_sales fs
JOIN    analytics.dim_date dd USING (date_key)
GROUP BY dd.year, dd.month
ORDER BY dd.year, dd.month;
'''
monthly = pd.read_sql(sql, engine)
monthly.assign(period=lambda d: d['year'].astype(str) + '-' + d['month'].astype(str).str.zfill(2)).set_index('period')[['revenue_brl']].plot(figsize=(10, 4), title='Monthly revenue (BRL)')

## Top product categories

In [ ]:
sql = '''
SELECT  dp.category_name_en                     AS category,
        COUNT(*)                                AS line_items,
        ROUND(SUM(fs.unit_price)::numeric, 2)   AS revenue_brl
FROM    analytics.fact_sales fs
JOIN    analytics.dim_product dp USING (product_key)
WHERE   dp.category_name_en IS NOT NULL
GROUP BY dp.category_name_en
ORDER BY revenue_brl DESC
LIMIT   15;
'''
pd.read_sql(sql, engine)

## Delivery performance

In [ ]:
sql = '''
SELECT  ROUND(AVG(delivery_days_actual), 1)     AS avg_actual_days,
        ROUND(AVG(delivery_days_estimated), 1)  AS avg_estimated_days,
        ROUND(100.0 * AVG(CASE WHEN delivery_days_actual <= delivery_days_estimated THEN 1 ELSE 0 END), 1) AS on_time_pct
FROM    analytics.fact_sales
WHERE   delivery_days_actual IS NOT NULL
AND     delivery_days_estimated IS NOT NULL;
'''
pd.read_sql(sql, engine)

## Latest data quality run

In [ ]:
sql = '''
SELECT  severity, status, COUNT(*) AS n
FROM    analytics.data_quality_log
WHERE   checked_at = (SELECT MAX(checked_at) FROM analytics.data_quality_log)
GROUP BY severity, status
ORDER BY severity;
'''
pd.read_sql(sql, engine)